## Classify Romansh vs. non-Romansh Text
This script illustrates how we can use the romansh_lemmatizer to distinguish between Romansh and non-Romansh text.

### Preliminaries: Import

In [33]:
# 1. Imports and initialisation
from romansh_lemmatizer import Lemmatizer

### Analyzing the Text
Define the example texts, then initialise the lemmatizer (non idiom-specific because we want to classify the text) and define a function that helps us classify the text.

The function computes an average coverage score by averaging the individual idiom scores. It also computes a global coverage score by counting how many tokens are in *any* Romansh variety.

In [34]:
# 2. Define example texts
samples = {
    "Romansh (Surmiran)": "La mamma fa paun e la figlia legia en casa.",
    "Romansh (Puter)": "La mamma fa paun e la figlia legia in chesa.",
    "German": "Die Mutter backt Brot und die Tochter liest im Haus.",
    "French": "La mère fait du pain et la fille lit dans la maison.",
    "Italian": "La madre fa il pane e la figlia legge in casa.",
}

In [35]:
# 3. Function to classify text
lemmatizer = Lemmatizer()

def classify_romansh_vs_nonromansh(text):
    doc = lemmatizer(text)

    # Get all idiom scores (coverage ratios)
    idiom_scores = doc.idiom_scores
    # Compute Romansh coverage: mean or sum over all idioms
    romansh_score = sum(idiom_scores.values()) / len(idiom_scores)

    # For absolute coverage, we can also count how many tokens matched any idiom
    total_tokens = len(doc.tokens)
    romansh_hits = sum(1 for tok in doc.tokens if any(tok.lemmas))
    coverage = romansh_hits / total_tokens if total_tokens else 0.0

    # Classify as Romansh if enough tokens are recognized
    label = "Romansh" if (coverage > 0.6 and romansh_score > 0.7) else "Non-Romansh"

    # Get the idiom with the highest individual score
    max_score = max(idiom_scores.values())
    best_items = [(k, v) for k, v in idiom_scores.items() if v == max_score]


    max_score = max(idiom_scores.values())
    best_items = [(k, v) for k, v in idiom_scores.items() if v == max_score]

    return {
        "avg_coverage": romansh_score,
        "global_coverage": coverage,
        "label": label,
        "best_idioms": [idiom.value for idiom, _ in best_items],
        "best_idiom_scores": [score for _, score in best_items],
    }


### Run Classification and Pretty Print Results
Here, we run the previously defined function and print the results in a nice format.

In [36]:
# Run classification
print("Romansh vs Non-Romansh classification\n")

for name, text in samples.items():
    result = classify_romansh_vs_nonromansh(text)

    print(f"Text: {name}")
    print(f"  → Label: {result['label']}")
    print(f"  → Romansh coverage (any idiom): {result['global_coverage']:.3f}")
    print(f"  → Averaged per-idiom Romansh coverage: {result['avg_coverage']:.3f}")

    # Handle multiple best idioms
    idioms = result["best_idioms"]
    scores = result["best_idiom_scores"]

    if len(idioms) == 1:
        print(f"  → Closest idiom: {idioms[0]} ({scores[0]:.3f})\n")
    else:
        print("  → Closest idioms:")
        for idiom, score in zip(idioms, scores):
            print(f"     - {idiom} ({score:.3f})")
        print()


Romansh vs Non-Romansh classification

Text: Romansh (Surmiran)
  → Label: Romansh
  → Romansh coverage (any idiom): 0.818
  → Averaged per-idiom Romansh coverage: 0.742
  → Closest idioms:
     - rm-rumgr (0.818)
     - rm-surmiran (0.818)
     - rm-puter (0.818)

Text: Romansh (Puter)
  → Label: Romansh
  → Romansh coverage (any idiom): 0.909
  → Averaged per-idiom Romansh coverage: 0.727
  → Closest idiom: rm-puter (0.909)

Text: German
  → Label: Non-Romansh
  → Romansh coverage (any idiom): 0.273
  → Averaged per-idiom Romansh coverage: 0.227
  → Closest idioms:
     - rm-sursilv (0.273)
     - rm-sutsilv (0.273)
     - rm-puter (0.273)
     - rm-vallader (0.273)

Text: French
  → Label: Non-Romansh
  → Romansh coverage (any idiom): 0.538
  → Averaged per-idiom Romansh coverage: 0.410
  → Closest idiom: rm-puter (0.538)

Text: Italian
  → Label: Non-Romansh
  → Romansh coverage (any idiom): 0.667
  → Averaged per-idiom Romansh coverage: 0.597
  → Closest idiom: rm-surmiran (0.667)